# One-Loop Power Spectra with `clax.ept`

Reproduces the plots from the CLASS-PT `nonlinear_pt.ipynb` notebook
using `clax.ept` (JAX-native, differentiable EFTofLSS one-loop power spectra).

In [1]:
import numpy as np

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import clax
from clax.ept import (
    EPTPrecisionParams, compute_ept, ept_kgrid,
    pk_mm_real, pk_gm_real, pk_gg_real,
    pk_mm_l0, pk_mm_l2, pk_mm_l4,
    pk_gg_l0, pk_gg_l2, pk_gg_l4,
    compute_ept_from_clax,
)

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "cm",
    "font.family": "STIXGeneral",
    "font.sans-serif": "Computer Modern",
    "font.size": 24})
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 6.0
plt.rcParams['ytick.major.size'] = 6.0
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0
plt.rcParams['xtick.minor.size'] = 4.0
plt.rcParams['ytick.minor.size'] = 4.0
plt.rcParams['xtick.minor.width'] = 0.8
plt.rcParams['ytick.minor.width'] = 0.8
plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['font.size'] = 24
plt.rcParams['axes.linewidth'] = 1.5

## 1. Cosmology and Pipeline Setup

Same cosmological parameters as the CLASS-PT notebook (Planck 2018 best-fit with one massive neutrino).

In [2]:
z_pk = 0.61

# Cosmological parameters matching CLASS-PT notebook
params = clax.CosmoParams(
    h=0.6736,
    omega_b=0.02237,
    omega_cdm=0.12,
    ln10A_s=np.log(2.089e-9 * 1e10),  # A_s = 2.089e-9
    n_s=0.9649,
    tau_reio=0.052,
    # Neutrinos: N_ur=2.0328, 1 massive with m=0.06 eV
)

# Run the clax pipeline: background + thermodynamics + perturbations
prec = clax.PrecisionParams()
bg = clax.background_solve(params, prec)
th = clax.thermodynamics_solve(params, prec, bg)

h = float(params.h)
print(f"h = {h}")
print(f"Omega_m = {bg.Omega_b + bg.Omega_cdm + bg.Omega_ncdm:.5f}")

h = 0.6736
Omega_m = 0.31519


## 2. Compute EPT One-Loop Spectra

Compute the linear P(k) at z=0.61 and run the EPT pipeline with and without IR resummation.

In [ ]:
# EPT k-grid
ept_prec = EPTPrecisionParams(ir_resummation=True)
ept_prec_noir = EPTPrecisionParams(ir_resummation=False)
k_h = ept_kgrid(ept_prec)  # h/Mpc
k_mpc = k_h * h              # Mpc^-1

# Extend the perturbation solve range to cover the full EPT k-grid,
# matching what CLASS-PT does (no extrapolation).
from dataclasses import replace as dc_replace
ept_kmax_mpc = float(k_mpc[-1])  # ~67 Mpc^-1
prec_ept = dc_replace(prec, pt_k_max_pk=ept_kmax_mpc * 1.01)

# Get linear P(k) at z_pk on the EPT k-grid
pk_table = clax.compute_pk_table(params, prec_ept, z=z_pk, k_eval=k_mpc)
pk_lin_mpc = np.array(pk_table.pk())  # Mpc^3
pk_lin_h = pk_lin_mpc * h**3           # (Mpc/h)^3

# Growth rate f from background spline
loga_z = float(jnp.log(1.0 / (1.0 + z_pk)))
f_z = float(bg.f_of_loga.evaluate(jnp.atleast_1d(loga_z))[0])
print(f"f(z={z_pk}) = {f_z:.4f}")

# Compute EPT with and without IR resummation
ept_ir = compute_ept(jnp.array(pk_lin_h), jnp.array(k_h), h=h, f=f_z, prec=ept_prec)
ept_noir = compute_ept(jnp.array(pk_lin_h), jnp.array(k_h), h=h, f=f_z, prec=ept_prec_noir)

# k-vector for plotting (h/Mpc)
kvec = np.array(k_h)

print("EPT computation complete.")
print(f"  k range: [{kvec[0]:.1e}, {kvec[-1]:.1e}] h/Mpc")
print(f"  N_k = {len(kvec)}")
print(f"  Perturbation solve extended to {ept_kmax_mpc:.1f} Mpc^-1")

In [ ]:
# Nuisance parameters (same as CLASS-PT notebook)
b1 = 2.0
cs = 1.0        # (Mpc/h)^2
b2 = -1.0
bG2 = 0.1
bGamma3 = -0.1
Pshot = 5e3     # (Mpc/h)^3
cs0 = 5.0       # (Mpc/h)^2
cs2 = 15.0      # (Mpc/h)^2
cs4 = -5.0      # (Mpc/h)^2
b4 = 100.0      # (Mpc/h)^4

# Compute assembled spectra from EPT components (IR-resummed)
pk_mm_full_ir = np.array(pk_mm_real(ept_ir, cs))
pk_gg = np.array(pk_gg_real(ept_ir, b1, b2, bG2, bGamma3, cs, cs0, Pshot))
pk_gm = np.array(pk_gm_real(ept_ir, b1, b2, bG2, bGamma3, cs, cs0))

# Without IR resummation
pk_mm_full = np.array(pk_mm_real(ept_noir, cs))

# RSD matter multipoles
pk_m0 = np.array(pk_mm_l0(ept_ir, cs0))
pk_m2 = np.array(pk_mm_l2(ept_ir, cs2))
pk_m4 = np.array(pk_mm_l4(ept_ir, cs4))

# RSD galaxy multipoles
pk_g0 = np.array(pk_gg_l0(ept_ir, b1, b2, bG2, bGamma3, cs0, Pshot, b4))
pk_g2 = np.array(pk_gg_l2(ept_ir, b1, b2, bG2, bGamma3, cs2, b4))
pk_g4 = np.array(pk_gg_l4(ept_ir, b1, b2, bG2, bGamma3, cs4, b4))

# Individual components
pk_tree = np.array(ept_noir.Pk_tree)
pk_loop = np.array(ept_noir.Pk_loop)
pk_ctr = 2.0 * cs * np.array(ept_noir.Pk_ctr)

print("All spectra computed.")

## 3. Real-Space Matter Power Spectrum (Plot 1)

Decomposition into tree-level (linear), 1-loop, and counterterm contributions.

In [ ]:
fig, ax = plt.subplots()
ax.loglog(kvec, np.abs(pk_loop), color='purple', ls='-', label='1-loop')
ax.loglog(kvec, np.abs(pk_ctr), color='b', ls='-', label='Counterterm')
ax.loglog(kvec, pk_tree, color='darkorange', ls='-', label='Linear')
ax.loglog(kvec, pk_mm_full, color='darkgreen', ls='-', label='Total')
ax.set_xlim(1e-3, 1)
ax.set_ylim(1, 5e4)
ax.set_xlabel(r'$k \; [h\,\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$P(k) \; [h^{-1}\mathrm{Mpc}]^3$')
ax.legend(fontsize=12, loc='upper left')
fig.tight_layout()
fig.savefig('figures/real_Pk.pdf')

## 4. IR Resummation Effect (Plot 2)

BAO wiggles are damped by the IR resummation, matching the observed broadening of the BAO peak.

In [ ]:
fig, ax = plt.subplots()
ax.plot(kvec, pk_lin_h * kvec**1.5, color='purple', ls='-.', label='Linear')
ax.plot(kvec, pk_mm_full * kvec**1.5, color='b', ls='--', label='1-loop, no IR resummation')
ax.plot(kvec, pk_mm_full_ir * kvec**1.5, color='r', ls='-', label='1-loop, IR resummation')
ax.set_xlim(1e-3, 0.5)
ax.set_ylim(55, 125)
ax.set_xlabel(r'$k \; [h\,\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$P(k)\,k^{3/2} \; [h^{-1}\mathrm{Mpc}]^{3/2}$')
ax.legend(fontsize=12, loc='upper right')
fig.tight_layout()
fig.savefig('figures/real_Pk_IR.pdf')

## 5. Real-Space Galaxy and Cross Spectra (Plot 3)

In [ ]:
fig, ax = plt.subplots()
ax.plot(kvec, pk_gg * kvec, color='b', ls='-', label='galaxy-galaxy')
ax.plot(kvec, pk_gm * kvec, color='darkorange', ls='-', label='galaxy-matter')
ax.plot(kvec, pk_mm_full_ir * kvec, color='purple', ls='-', label='matter-matter')
ax.set_xlim(1e-3, 0.3)
ax.set_ylim(1, 2200)
ax.set_xlabel(r'$k \; [h\,\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$P(k)\,k \; [h^{-1}\mathrm{Mpc}]^{2}$')
ax.legend(fontsize=12, loc='upper right')
fig.tight_layout()
fig.savefig('figures/real_Pkgg.pdf')

## 6. Bias Spectrum Components (Plot 4)

Individual loop integral contributions to the biased tracer power spectrum.

In [ ]:
# Extract bias cross-spectra from the IR-resummed EPT result
pk_Id2 = np.array(ept_ir.Pk_Id2)
pk_IG2 = np.array(ept_ir.Pk_IG2)
pk_Id2d2 = np.array(ept_ir.Pk_Id2d2)
pk_Id2G2 = np.array(ept_ir.Pk_Id2G2)
pk_IG2G2 = np.array(ept_ir.Pk_IG2G2)
pk_IFG2 = np.array(ept_ir.Pk_IFG2)

fig, ax = plt.subplots()
ax.loglog(kvec, pk_lin_h, color='k', ls='-', label=r'$P_{\rm lin}$')
ax.loglog(kvec, np.abs(pk_Id2), color='b', ls='-', label=r'$\mathcal{I}_{\delta^2}$')
ax.loglog(kvec, np.abs(pk_IG2), color='darkorange', ls='--', label=r'$\mathcal{I}_{\mathcal{G}_2}$')
ax.loglog(kvec, np.abs(pk_Id2d2), color='r', ls='-.', label=r'$\mathcal{I}_{\delta^2\delta^2}$')
ax.loglog(kvec, np.abs(pk_Id2G2), color='teal', ls=':', label=r'$\mathcal{I}_{\delta^2\mathcal{G}_2}$')
ax.loglog(kvec, np.abs(pk_IG2G2), color='purple', ls='-', label=r'$\mathcal{I}_{\mathcal{G}_2\mathcal{G}_2}$')
ax.loglog(kvec, np.abs(pk_IFG2), color='darkgreen', ls='--', label=r'$\mathcal{F}_{\mathcal{G}_2}$')
ax.set_xlim(1e-3, 1)
ax.set_ylim(1, 5e4)
ax.set_xlabel(r'$k \; [h\,\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$P(k) \; [h^{-1}\mathrm{Mpc}]^3$')
ax.legend(fontsize=10, loc='upper left')
fig.tight_layout()
fig.savefig('figures/real_Pkgg_breakdown.pdf')

## 7. Redshift-Space Matter Multipoles (Plot 5)

In [ ]:
fig, ax = plt.subplots()
ax.plot(kvec, pk_m0 * kvec, color='darkgreen', ls='-', label=r'$\ell = 0$')
ax.plot(kvec, pk_m2 * kvec, color='darkorange', ls='-', label=r'$\ell = 2$')
ax.plot(kvec, pk_m4 * kvec, color='purple', ls='-', label=r'$\ell = 4$')
ax.set_xlim(1e-3, 0.3)
ax.set_ylim(1, 1000)
ax.set_xlabel(r'$k \; [h\,\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$P_{\ell,\,mm}(k)\,k \; [h^{-1}\mathrm{Mpc}]^{2}$')
ax.legend(fontsize=12, loc='upper right')
fig.tight_layout()
fig.savefig('figures/rsd_Pkmm.pdf')

## 8. Redshift-Space Galaxy Multipoles (Plot 6)

In [ ]:
fig, ax = plt.subplots()
ax.plot(kvec, pk_g0 * kvec, color='darkgreen', ls='-', label=r'$\ell = 0$')
ax.plot(kvec, pk_g2 * kvec, color='darkorange', ls='-', label=r'$\ell = 2$')
ax.plot(kvec, pk_g4 * kvec, color='purple', ls='-', label=r'$\ell = 4$')
ax.set_xlim(1e-3, 0.3)
ax.set_ylim(1, 2500)
ax.set_xlabel(r'$k \; [h\,\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$P_{\ell,\,gg}(k)\,k \; [h^{-1}\mathrm{Mpc}]^{2}$')
ax.legend(fontsize=12, loc='upper right')
fig.tight_layout()
fig.savefig('figures/rsd_Pkgg.pdf')

## 9. Position-Space Correlation Function (Plot 7)

Fourier transform P(k) to xi(r) using FFTLog, comparing linear, 1-loop without IR resummation, and 1-loop with IR resummation.

In [ ]:
from scipy.special import gamma as sp_gamma
from scipy.interpolate import InterpolatedUnivariateSpline

def pk_to_xi(kvec_h, pk_h, Nmax=256, kmin=1e-4, kmax=100.0,
             rmin=0.01, rmax=1000.0, bk=-1.1001):
    """Convert P(k) to xi(r) via FFTLog (Hankel transform).

    Uses the same algorithm as the CLASS-PT notebook.
    """
    Delta = np.log(kmax / kmin) / (Nmax - 1)
    Delta_r = np.log(rmax / rmin) / (Nmax - 1)

    kbins = kmin * np.exp(Delta * np.arange(Nmax))
    rtab = rmin * np.exp(Delta_r * np.arange(Nmax))

    # Interpolate P(k) to the FFTLog k-grid
    pk_interp = InterpolatedUnivariateSpline(np.log(kvec_h), np.log(np.abs(pk_h) + 1e-30))
    pk_fftlog = np.sign(np.interp(kbins, kvec_h, pk_h)) * np.exp(pk_interp(np.log(kbins)))

    # Discretize with bias and UV cutoff
    Pdiscrin = pk_fftlog * np.exp(-(kbins / 4.0)**4 - bk * np.arange(Nmax) * Delta)

    # FFT and symmetrize
    cm = np.fft.fft(Pdiscrin) / Nmax
    jsNm = np.arange(-Nmax // 2, Nmax // 2 + 1, 1)
    etam = bk + 2j * np.pi * jsNm / Nmax / Delta

    cmsym = np.zeros(Nmax + 1, dtype=complex)
    for i in range(Nmax + 1):
        if (i + 2 - Nmax // 2) < 1:
            cmsym[i] = kmin**(-etam[i]) * np.conj(cm[-i + Nmax // 2])
        else:
            cmsym[i] = kmin**(-etam[i]) * cm[i - Nmax // 2]
    cmsym[0] /= 2
    cmsym[-1] /= 2

    # Kernel J_0(r, nu) for spherical Bessel j_0
    def J0(r, nu):
        return -np.sin(np.pi * nu / 2) * r**(-3 - nu) * sp_gamma(2 + nu) / (2 * np.pi**2)

    # Sum over modes
    xi = np.zeros(Nmax)
    for i in range(Nmax):
        for j in range(Nmax + 1):
            xi[i] += np.real(cmsym[j] * J0(rtab[i], etam[j]))

    return rtab, xi

# Compute xi(r) for three cases
rtab, xi_lin = pk_to_xi(kvec, pk_lin_h)
_, xi_loop = pk_to_xi(kvec, pk_mm_full)
_, xi_loop_ir = pk_to_xi(kvec, pk_mm_full_ir)

# Interpolate to plotting grid
rvec = np.logspace(0, np.log10(3), 1000)
rvec = 10**rvec

xi_lin_interp = InterpolatedUnivariateSpline(rtab, xi_lin)
xi_loop_interp = InterpolatedUnivariateSpline(rtab, xi_loop)
xi_loop_ir_interp = InterpolatedUnivariateSpline(rtab, xi_loop_ir)

fig, ax = plt.subplots()
ax.plot(rvec, [xi_lin_interp(r) * r for r in rvec], color='purple', ls='-.', label='Linear')
ax.plot(rvec, [xi_loop_interp(r) * r for r in rvec], color='b', ls='--', label='1-loop, no IR resummation')
ax.plot(rvec, [xi_loop_ir_interp(r) * r for r in rvec], color='red', ls='-', label='1-loop, IR resummation')
ax.set_xlim(60, 140)
ax.set_ylim(-0.03, 0.15)
ax.set_xlabel(r'$r \; [h^{-1}\mathrm{Mpc}]$')
ax.set_ylabel(r'$r \cdot \xi(r) \; [h^{-1}\mathrm{Mpc}]$')
ax.legend(fontsize=12, loc='upper left')
fig.tight_layout()
fig.savefig('figures/real_Xi.pdf')

## 10. External Linear Power Spectrum (Plot 10)

Demonstrates using an arbitrary external P_lin(k) as input to `compute_ept`.

In [ ]:
# The EPT computation already used an "external" P_lin via compute_pk_table.
# Here we show this explicitly: any P_lin(k) array can be fed to compute_ept.

# Reuse the same pk_lin_h from above as the "external" input
ept_ext = compute_ept(jnp.array(pk_lin_h), jnp.array(k_h), h=h, f=f_z, prec=ept_prec)

pk_gg_ext = np.array(pk_gg_real(ept_ext, b1, b2, bG2, bGamma3, cs, cs0, Pshot))
pk_gm_ext = np.array(pk_gm_real(ept_ext, b1, b2, bG2, bGamma3, cs, cs0))
pk_mm_ext = np.array(pk_mm_real(ept_ext, cs))

fig, ax = plt.subplots()
ax.loglog(kvec, pk_gg_ext, color='r', ls='-', label='galaxy-galaxy')
ax.loglog(kvec, pk_gm_ext, color='g', ls='-', label='galaxy-matter')
ax.loglog(kvec, pk_mm_ext, color='b', ls='-', label='matter-matter')
ax.loglog(kvec, pk_lin_h, color='purple', ls='--', label='input linear P(k)')
ax.set_xlim(1e-3, 1)
ax.set_ylim(100, 1e5)
ax.set_xlabel(r'$k \; [h\,\mathrm{Mpc}^{-1}]$')
ax.set_ylabel(r'$P(k) \; [h^{-1}\mathrm{Mpc}]^3$')
ax.legend(fontsize=12, loc='lower left')
fig.tight_layout()
fig.savefig('figures/external_Pk.pdf')

## 11. CMB Lensing: Nonlinear Corrections (Plots 8-9)

Compute lensed TT and lensing potential $C_\ell^{\phi\phi}$ ratios: nonlinear (PT and Halofit) vs linear.

**Note:** This requires a full Boltzmann perturbation solve (expensive) for `source_lens`.

In [ ]:
from clax.perturbations import perturbations_solve
from clax.lensing import (
    compute_cl_pp, compute_cl_pp_nonlinear,
    compute_nl_correction_halofit, lens_cls,
)
from clax.harmonic import compute_cls_all_fast
from dataclasses import replace as dc_replace

# Use planck_cl preset for science-grade accuracy
prec_cl = clax.PrecisionParams.planck_cl()
bg_cl = clax.background_solve(params, prec_cl)
th_cl = clax.thermodynamics_solve(params, prec_cl, bg_cl)

print("Running full Boltzmann perturbation solve (this may take a few minutes)...")
pt_cl = perturbations_solve(params, prec_cl, bg_cl, th_cl)
print("Done.")

In [ ]:
l_max = 2500
l_values = list(range(2, l_max + 1))

# Compute unlensed C_l
print("Computing unlensed C_l...")
cl_tt, cl_ee, cl_te, cl_bb = compute_cls_all_fast(pt_cl, params, bg_cl, l_max=l_max)

# Compute C_l^pp: linear, Halofit
print("Computing C_l^pp (linear)...")
cl_pp_lin = compute_cl_pp(pt_cl, params, bg_cl, l_values)

print("Computing C_l^pp (Halofit)...")
cl_pp_halofit = compute_cl_pp_nonlinear(
    pt_cl, params, bg_cl, l_values, method="halofit", z_ref=0.0)

# Pad to l_max+1 for lens_cls (needs arrays indexed by l)
cl_pp_lin_full = np.zeros(l_max + 1)
cl_pp_lin_full[2:] = np.array(cl_pp_lin)
cl_pp_halofit_full = np.zeros(l_max + 1)
cl_pp_halofit_full[2:] = np.array(cl_pp_halofit)

# Lens the C_l with linear and Halofit C_l^pp
print("Lensing C_l (linear C_l^pp)...")
tt_lensed_lin, ee_lensed_lin, te_lensed_lin, bb_lensed_lin = lens_cls(
    cl_tt, cl_ee, cl_te, cl_bb, jnp.array(cl_pp_lin_full), l_max=l_max)

print("Lensing C_l (Halofit C_l^pp)...")
tt_lensed_hf, ee_lensed_hf, te_lensed_hf, bb_lensed_hf = lens_cls(
    cl_tt, cl_ee, cl_te, cl_bb, jnp.array(cl_pp_halofit_full), l_max=l_max)

print("All lensing computations complete.")

In [ ]:
ll = np.arange(2, l_max + 1)

# Plot 8: Lensed TT ratio
tt_lin = np.array(tt_lensed_lin)[2:]
tt_hf = np.array(tt_lensed_hf)[2:]

fig, ax = plt.subplots()
ax.semilogx(ll, tt_hf / tt_lin, ls='-', label='Halofit')
ax.set_xlim(500, 2500)
ax.set_ylim(0.998, 1.005)
ax.set_xlabel(r'Angular Multipole $\ell$')
ax.set_ylabel(r'$C_\ell^{\mathrm{(TT),\,non-linear}} / C_\ell^{\mathrm{(TT),\,linear}}$')
ax.legend(loc='upper left')
fig.tight_layout()
fig.savefig('figures/ratios-cltt.pdf')

# Plot 9: Lensing potential ratio
pp_lin = np.array(cl_pp_lin)
pp_hf = np.array(cl_pp_halofit)

fig, ax = plt.subplots()
ax.semilogx(ll, pp_hf / pp_lin, ls='-', label='Halofit')
ax.set_xlim(10, 2500)
ax.set_ylim(0.95, 1.3)
ax.set_xlabel(r'Angular Multipole $\ell$')
ax.set_ylabel(r'$C_\ell^{(\phi\phi),\,\mathrm{non-linear}} / C_\ell^{(\phi\phi),\,\mathrm{linear}}$')
ax.legend(loc='upper left')
fig.tight_layout()
fig.savefig('figures/ratios-clpp.pdf')